# Scenario: University Library Assistant
 A large university library has thousands of digitized textbooks, research papers, and course notes. Students often struggle to find specific explanations or summaries when preparing for exams. Instead of manually searching through PDFs, the library deploys a RAG chatbot that acts like a study companion.
 How It Works
 - Input Source: Students upload or access a textbook PDF (e.g., Introduction_to_Data_Science.pdf).
 - Chunking: The chatbot splits the textbook into smaller sections so that each concept is searchable.
 - Embeddings + Vector DB: Each section is embedded and stored in Chroma, making the textbook searchable by meaning.
 - Retriever: When a student asks, “Explain the difference between supervised and unsupervised learning,” the retriever pulls the most relevant sections.
 - LLM Response: The Hugging Face model generates a clear, concise answer tailored to the student’s query.
 - Interactive Chat: Students can keep asking follow-up questions, turning the textbook into a conversational tutor.

In [ ]:
# UNIVERSITY LIBRARY ASSISTANT (RAG CHATBOT)
# Study Companion for Students

# STEP 0 — Install Required Libraries


!pip install -q chromadb sentence-transformers pypdf transformers accelerate

# STEP 1 — Import Libraries

import os
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from google.colab import files
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


# ----------------------------------------------------------
# STEP 2 — Upload Textbook PDF
# ----------------------------------------------------------

print("Upload the textbook PDF (example: Introduction_to_Data_Science.pdf)")

uploaded = files.upload()

pdf_file = list(uploaded.keys())[0]

print("Loading textbook...")

reader = PdfReader(pdf_file)

text = ""

for page in reader.pages:
    extracted = page.extract_text()
    if extracted:
        text += extracted

print("Textbook Loaded")
print("Total characters:", len(text))

print("\nPreview:\n")
print(text[:500])


# STEP 3 — Chunk the Document

print("\nSplitting textbook into sections...")

def chunk_text(text, chunk_size=700, overlap=100):

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


chunks = chunk_text(text)

print("Total Sections Created:", len(chunks))
print("\nExample Section:\n")
print(chunks[0])


# STEP 4 — Load Embedding Model

print("\nLoading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")


# STEP 5 — Create Vector Database

client = chromadb.Client()

try:
    client.delete_collection("library_collection")
except:
    pass

collection = client.create_collection("library_collection")

print("Vector database created")


# STEP 6 — Store Sections in Vector DB

print("\nCreating embeddings and storing sections...")

for i, chunk in enumerate(chunks):

    embedding = embedding_model.encode(chunk).tolist()

    collection.add(
        documents=[chunk],
        embeddings=[embedding],
        ids=[str(i)]
    )

print("All textbook sections stored successfully")


# STEP 7 — Retriever Function

def retrieve(query, k=3):

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    return results["documents"][0]


# STEP 8 — Load LLM

print("\nLoading Study Assistant Model...")

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

print("Model loaded successfully")


# STEP 9 — Question Answering Function

def answer_question(query):

    context_docs = retrieve(query)

    context = " ".join(context_docs)

    prompt = f"""
You are a helpful university study assistant.

Use the textbook context below to answer the student's question.
Explain the concept clearly and simply for exam preparation.

If the answer is not in the text, say:
"Not found in the textbook."

Context:
{context}

Question:
{query}

Answer:
"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.3
    )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return answer


# STEP 10 — Student Chat Loop

print("\n==============================")
print("University Library Assistant Ready")
print("Type 'exit' to stop")
print("==============================\n")

while True:

    question = input("Ask your study question: ")

    if question.lower() == "exit":
        print("Good luck with your studies!")
        break

    answer = answer_question(question)

    print("\nAnswer:\n", answer)
    print("\n" + "-"*60 + "\n")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.4/333.4 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently

Saving Introduction_to_Data_Science.pdf to Introduction_to_Data_Science.pdf
Loading textbook...
Textbook Loaded
Total characters: 3427

Preview:

Introduction to Data Science
Data Science is an interdisciplinary field that combines statistics, computer science, and domain
knowledge to extract meaningful insights from data. In today's digital world, organizations generate
massive amounts of data from websites, mobile applications, sensors, financial systems, and social
media platforms. Data scientists analyze this data to help businesses make better decisions.
1. What is Data Science?
Data Science involves collecting, cleaning, analyzing, 

Splitting textbook into sections...
Total Sections Created: 6

Example Section:

Introduction to Data Science
Data Science is an interdisciplinary field that combines statistics, computer science, and domain
knowledge to extract meaningful insights from data. In today's digital world, organizations generate
massive amounts of data from websites, mobi

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded successfully

University Library Assistant Ready
Type 'exit' to stop

Ask your study question: what is supervised learning


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Answer:
 Models are trained using labeled data where the correct output is known. Examples include predicting house prices or classifying emails as spam or not spam.

------------------------------------------------------------

Ask your study question: what is data science

Answer:
 Data Science is an interdisciplinary field that combines statistics, computer science, and domain knowledge to extract meaningful insights from data.

------------------------------------------------------------

